In [2]:
# ========Setup & Imports =========
import sys
import json
from pathlib import Path
from typing import Literal
from pydantic import BaseModel, ValidationError
import pandas as pd


root_path = Path.cwd().parent if Path.cwd().name in ["notebooks"] else Path.cwd()
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

from src.client import client


In [3]:
# ========Part A: Dataset & Pydantic Schema =========
class SentimentAnalysis(BaseModel):
    reasoning: str
    label: Literal["positive", "negative", "neutral"]

class SentimentAnalysisReversed(BaseModel):
    label: Literal["positive", "negative", "neutral"]
    reasoning: str

reviews = [
    "The battery life is stellar, lasting two full days easily.",
    "Customer service was utterly unhelpful and rude.",
    "The camera is decent, but the software has frequent glitches.",
    "Not worth the ticket, though the visual effects were neat.",
    "It arrived on time."
]


In [5]:
# ========Part A: The 3 Extraction Levels =========

system_instruction = (
    "Analyze the sentiment of the review. "
    "Allowed labels: positive, negative, neutral."
)

strict_schema = {
    "type": "object",
    "properties": {
        "reasoning": {"type": "string"},
        "label": {
            "type": "string",
            "enum": ["positive", "negative", "neutral"]
        }
    },
    "required": ["reasoning", "label"],
    "additionalProperties": False
}

def prompt_level1(review):
    return (
        f"{system_instruction}\n"
        "Return JSON with the keys 'reasoning' and 'label'.\n"
        f"Review: {review}"
    )

def validate_json(raw_text):
    try:
        cleaned = raw_text.strip()
        cleaned = cleaned.replace("```json", "").replace("```", "").strip()

        parsed = json.loads(cleaned)
        SentimentAnalysis.model_validate(parsed)
        return True
    except (json.JSONDecodeError, ValidationError, TypeError):
        return False

results_levels = []

for review in reviews:
    # 1. Prompt-only
    res1 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt_level1(review)}
        ],
        temperature=0
    ).choices[0].message.content or ""

    # 2. JSON Mode
    res2 = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": (
                    system_instruction
                    + " Return a JSON object with 'reasoning' and 'label'."
                )
            },
            {"role": "user", "content": review}
        ],
        temperature=0
    ).choices[0].message.content or ""

    # 3. Strict JSON Schema
    res3 = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "sentiment_response",
                "strict": True,
                "schema": strict_schema
            }
        },
        messages=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": review}
        ],
        temperature=0
    ).choices[0].message.content or ""

    results_levels.append({
        "review": review,
        "prompt_only_valid": validate_json(res1),
        "json_mode_valid": validate_json(res2),
        "strict_schema_valid": validate_json(res3),
        "prompt_only_output": res1,
        "json_mode_output": res2,
        "strict_schema_output": res3
    })

df_levels = pd.DataFrame(results_levels)
display(df_levels)
df_levels.to_csv(root_path/"report"/"ImgPlt"/"04_structured_tools.csv", index=False)

,review,prompt_only_valid,json_mode_valid,strict_schema_valid,prompt_only_output,json_mode_output,strict_schema_output
0,"The battery life is stellar, lasting two full ...",True,True,True,"{\n ""reasoning"": ""The review expresses a stro...","{\n ""reasoning"": ""The review expresses a posi...","{""reasoning"":""The review expresses a positive ..."
1,Customer service was utterly unhelpful and rude.,True,True,True,"{\n ""reasoning"": ""The review expresses dissat...","{\n ""reasoning"": ""The review expresses dissat...","{""reasoning"":""The review expresses dissatisfac..."
2,"The camera is decent, but the software has fre...",True,True,True,"{\n ""reasoning"": ""The review expresses a mixe...","{\n ""reasoning"": ""The review expresses a mixe...","{""reasoning"":""The review expresses a mixed sen..."
3,"Not worth the ticket, though the visual effect...",True,True,True,"{\n ""reasoning"": ""The review expresses disapp...","{\n ""reasoning"": ""The review expresses a nega...","{""reasoning"":""The review expresses a negative ..."
4,It arrived on time.,True,True,True,"{\n ""reasoning"": ""The review expresses a posi...","{\n ""reasoning"": ""The review states that the ...","{""reasoning"":""The review states that the item ..."


In [6]:
# ========Field Ordering Impact =========
tricky_review = "Not worth the ticket, though the visual effects were neat."

schema_reasoning_first = {
    "type": "object",
    "properties": {
        "reasoning": {"type": "string"},
        "label": {"type": "string", "enum": ["positive", "negative", "neutral"]}
    },
    "required": ["reasoning", "label"],
    "additionalProperties": False
}

schema_label_first = {
    "type": "object",
    "properties": {
        "label": {"type": "string", "enum": ["positive", "negative", "neutral"]},
        "reasoning": {"type": "string"}
    },
    "required": ["label", "reasoning"],
    "additionalProperties": False
}

res_reasoning_first = client.chat.completions.create(
    model="gpt-4o-mini",
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "sentiment_reasoning_first",
            "strict": True,
            "schema": schema_reasoning_first
        }
    },
    messages=[{"role": "user", "content": tricky_review}],
    temperature=0
).choices[0].message.content

res_label_first = client.chat.completions.create(
    model="gpt-4o-mini",
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "sentiment_label_first",
            "strict": True,
            "schema": schema_label_first
        }
    },
    messages=[{"role": "user", "content": tricky_review}],
    temperature=0
).choices[0].message.content

print("Reasoning First:\n", res_reasoning_first)
print("\nLabel First:\n", res_label_first)


Reasoning First:
 {"reasoning":"The statement expresses a negative sentiment overall, as it indicates that the experience (likely a movie or event) was not worth the ticket price, despite acknowledging some positive aspects (the visual effects). This suggests disappointment or dissatisfaction with the overall value.","label":"negative"}

Label First:
 {"label":"negative","reasoning":"The statement expresses disappointment by stating that the experience was 'not worth the ticket,' indicating dissatisfaction, despite acknowledging some positive aspects like the visual effects."}


In [8]:
# ========Part B: Local Tools Definition =========
def get_exchange_rate(currency: str) -> float:
    rates = {"USD": 60000.0, "EUR": 65000.0, "GBP": 76000.0}
    return rates.get(currency.upper(), -1.0)

def calculator(expression: str) -> float:
    allowed_chars = set("0123456789+-*/.() ")
    if not set(expression).issubset(allowed_chars):
        raise ValueError("Invalid mathematical expression")
    return float(eval(expression))

tools_definition = [
    {
        "type": "function",
        "function": {
            "name": "get_exchange_rate",
            "description": "Get current exchange rate to Tomans for a currency",
            "parameters": {
                "type": "object",
                "properties": {
                    "currency": {"type": "string", "description": "Currency code e.g. USD, EUR"}
                },
                "required": ["currency"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Calculate mathematical expression string",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "e.g. '50 * 60000'"}
                },
                "required": ["expression"]
            }
        }
    }
]

available_tools = {
    "get_exchange_rate": get_exchange_rate,
    "calculator": calculator
}


In [9]:
# ========Part B: Tool Execution Loop =========
user_query = "How much is 50 dollars in tomans, and what does 10 percent of that amount to?"

messages = [
    {"role": "system", "content": "You are a helpful assistant. Use provided tools for currency and math calculations."},
    {"role": "user", "content": user_query}
]

max_steps = 4
step = 0

while step < max_steps:
    step += 1
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools_definition,
        tool_choice="auto",
        temperature=0
    )

    msg = response.choices[0].message
    messages.append(msg)

    if not msg.tool_calls:
        print(f"Final Answer (Reached in step {step}):\n{msg.content}")
        break

    for tool_call in msg.tool_calls:
        func_name = tool_call.function.name

        try:
            func_args = json.loads(tool_call.function.arguments)
        except json.JSONDecodeError as e:
            content = f"Error: Invalid JSON arguments: {str(e)}"
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": content})
            continue

        if func_name in available_tools:
            try:
                func_to_call = available_tools[func_name]
                tool_output = func_to_call(**func_args)
                tool_content = str(tool_output)
            except Exception as e:
                tool_content = f"Execution error: {str(e)}"
        else:
            tool_content = f"Error: Tool '{func_name}' not found."

        print(f"Step {step}: Called '{func_name}' with {func_args} -> Output: {tool_content}")

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": tool_content
        })


Step 1: Called 'get_exchange_rate' with {'currency': 'USD'} -> Output: 60000.0
Step 2: Called 'calculator' with {'expression': '50 * 60000'} -> Output: 3000000.0
Step 3: Called 'calculator' with {'expression': '3000000 * 0.10'} -> Output: 300000.0
Final Answer (Reached in step 4):
50 dollars is equivalent to 3,000,000 tomans. Ten percent of that amount is 300,000 tomans.
